# Perhitungan Matriks Jarak dan Waktu Tempuh via OSRM

Notebook ini menghitung matriks jarak (meter) dan waktu tempuh (detik) antar seluruh
node model (depo, titik genangan, fasilitas perantara) menggunakan **OSRM Table
Service**, berdasarkan jaringan jalan aktual, bukan formula geometris (Manhattan/Haversine).

Notebook ini dijalankan **sekali secara offline**, bukan setiap kali server jalan.
Karena skenario banjir bersifat statis (Batasan Masalah proposal, Subbab 1.3), matriks
jarak tidak perlu dihitung ulang secara real-time. Hasilnya disimpan sebagai
`distance_matrix.npy` dan `time_matrix.npy`, lalu dimuat langsung oleh backend
(`app/main.py` sudah mendukung pemuatan file ini di `lifespan()`).

**Urutan node harus SAMA PERSIS dengan yang dipakai `instance.py`:**
`[0 .. n_depots) -> depo, [n_depots .. n_depots+n_floods) -> genangan, [.. n_total) -> IF`

**Input:**
- `depo.csv`, `floods.csv`, `if.csv` (skema sama seperti yang dipakai backend)

**Output:**
- `distance_matrix.npy` (n_total x n_total, meter)
- `time_matrix.npy` (n_total x n_total, detik)
- `osrm_matrix_report.csv` (ringkasan sel yang gagal/fallback, untuk transparansi di Bab 5)


In [8]:
# Install dependencies (jalankan sekali)
!pip install requests pandas numpy tqdm -q


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import requests
import pandas as pd
import numpy as np
import time
import json
from pathlib import Path
from tqdm import tqdm

# ===== KONFIGURASI =====
# Ganti ke server OSRM sendiri kalau punya (mis. "http://localhost:5000"),
# atau biarkan pakai demo publik untuk uji coba skala kecil.
OSRM_BASE = "https://router.project-osrm.org"

# Ukuran blok per request. Server demo publik OSRM membatasi ukuran tabel
# (default sekitar 100x100 pada beberapa mirror). Blok kecil = lebih banyak
# request tapi lebih kecil risiko ditolak/timeout.
BLOCK_SIZE = 40

# Jeda antar-request agar tidak kena rate limit (detik)
REQUEST_DELAY_S = 1.0

# Retry per blok jika gagal
MAX_RETRIES = 3
RETRY_WAIT_S = 5

DATA_DIR = Path("../backend/app/data")  # sesuaikan lokasi CSV backend kamu
OUTPUT_DIR = Path("output_osrm")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Setup selesai.")

Setup selesai.


## 1. Muat dan Susun Node Sesuai Urutan `instance.py`

Replikasi logika pembersihan dari `backend/app/main.py` (`_load_depots`,
`_load_floods`, `_load_ifs`) supaya urutan dan koordinat node identik dengan
yang dipakai backend saat runtime. Kalau urutan berbeda, matriks yang dihasilkan
tidak akan cocok dengan index yang dipakai algoritma.

In [10]:
SBY_LAT_MIN, SBY_LAT_MAX = -7.38, -7.13
SBY_LON_MIN, SBY_LON_MAX = 112.58, 112.87


def _bbox_filter(df, lat_col="lat", lon_col="lon"):
    mask = (
        (df[lat_col] >= SBY_LAT_MIN) & (df[lat_col] <= SBY_LAT_MAX)
        & (df[lon_col] >= SBY_LON_MIN) & (df[lon_col] <= SBY_LON_MAX)
    )
    return df[mask].reset_index(drop=True)


def load_depots(path):
    df = pd.read_csv(path)
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df = df.dropna(subset=["lat", "lon"]).reset_index(drop=True)
    return _bbox_filter(df)


def load_floods(path):
    df = pd.read_csv(path)
    df = df.rename(columns={"Latitude": "lat", "Longitude": "lon"})
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df = df.dropna(subset=["lat", "lon"]).reset_index(drop=True)
    return _bbox_filter(df)


def load_ifs(path):
    df = pd.read_csv(path)
    df = df.rename(columns={"latitude": "lat", "longitude": "lon"})
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df = df.dropna(subset=["lat", "lon"]).reset_index(drop=True)
    return _bbox_filter(df)


depots_df = load_depots(DATA_DIR / "depo.csv")
floods_df = load_floods(DATA_DIR / "floods-skenario1.csv")
ifs_df = load_ifs(DATA_DIR / "if.csv")

n_depots, n_floods, n_ifs = len(depots_df), len(floods_df), len(ifs_df)
n_total = n_depots + n_floods + n_ifs

print(f"Depo   : {n_depots}")
print(f"Genangan: {n_floods}")
print(f"IF     : {n_ifs}")
print(f"Total node: {n_total}")

lats = np.concatenate([depots_df["lat"].values, floods_df["lat"].values, ifs_df["lat"].values])
lons = np.concatenate([depots_df["lon"].values, floods_df["lon"].values, ifs_df["lon"].values])
coords = list(zip(lats, lons))

Depo   : 12
Genangan: 34
IF     : 140
Total node: 186


## 2. Fungsi Query OSRM Table Service

`GET /table/v1/driving/{lon,lat;lon,lat;...}?sources=...&destinations=...`

OSRM memakai urutan **lon,lat** (bukan lat,lon). Kita query per blok:
untuk tiap blok baris (`sources`), minta jarak ke SEMUA node (`destinations`),
lalu susun ulang jadi matriks penuh.

In [11]:
def build_coord_string(coord_list):
    return ";".join(f"{lon:.6f},{lat:.6f}" for lat, lon in coord_list)


def query_osrm_table(all_coords, source_idx, dest_idx, max_retries=MAX_RETRIES):
    """Query OSRM table untuk blok source_idx x dest_idx.

    Mengirim SELURUH koordinat sebagai satu daftar titik, lalu memakai
    parameter sources/destinations (indeks) supaya tidak perlu membangun
    string koordinat berulang per pasangan blok.
    """
    coords_str = build_coord_string(all_coords)
    sources_str = ";".join(str(i) for i in source_idx)
    dest_str = ";".join(str(i) for i in dest_idx)
    url = (
        f"{OSRM_BASE}/table/v1/driving/{coords_str}"
        f"?sources={sources_str}&destinations={dest_str}"
        f"&annotations=distance,duration"
    )

    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200:
                data = r.json()
                if data.get("code") == "Ok":
                    return data["distances"], data["durations"]
                print(f"  [WARN] OSRM code={data.get('code')} untuk blok {len(source_idx)}x{len(dest_idx)}")
            else:
                print(f"  [WARN] HTTP {r.status_code} percobaan {attempt}")
        except Exception as e:
            print(f"  [ERROR] percobaan {attempt}: {e}")
        time.sleep(RETRY_WAIT_S)

    return None, None


print("Fungsi query siap.")

Fungsi query siap.


## 3. Jalankan Query Blok demi Blok

Catatan penting: mengirim SEMUA koordinat pada tiap request (lewat parameter
`coords_str`) itu boros kalau `n_total` besar. Untuk skala node kamu saat ini
(~200 node) ini masih wajar. Kalau nanti node bertambah signifikan (>500),
pertimbangkan self-host OSRM supaya tidak bergantung pada batas server demo publik.

In [12]:
node_indices = list(range(n_total))
blocks = [node_indices[i:i + BLOCK_SIZE] for i in range(0, n_total, BLOCK_SIZE)]

dist_matrix = np.full((n_total, n_total), np.nan)
time_matrix = np.full((n_total, n_total), np.nan)
failed_blocks = []

total_pairs = len(blocks) * len(blocks)
pbar = tqdm(total=total_pairs, desc="Querying OSRM")

for src_block in blocks:
    for dst_block in blocks:
        distances, durations = query_osrm_table(coords, src_block, dst_block)
        if distances is not None:
            for si, s in enumerate(src_block):
                for di, d in enumerate(dst_block):
                    dist_matrix[s, d] = distances[si][di]
                    time_matrix[s, d] = durations[si][di]
        else:
            failed_blocks.append((src_block[0], dst_block[0]))
        time.sleep(REQUEST_DELAY_S)
        pbar.update(1)

pbar.close()
print(f"Blok gagal: {len(failed_blocks)} dari {total_pairs}")

Querying OSRM: 100%|██████████| 25/25 [00:52<00:00,  2.11s/it]

Blok gagal: 0 dari 25


## 4. Fallback untuk Sel yang Gagal

Kalau ada sel yang tidak terisi (server OSRM gagal/timeout untuk blok tertentu,
atau pasangan node memang tidak terhubung jaringan jalan sesuai constraint OSRM
seperti one-way), isi dengan formula Manhattan sebagai fallback, konsisten dengan
fallback yang sudah ada di `osrm.py` (`_manhattan_fallback`) untuk visualisasi rute.
Ini dicatat di laporan supaya transparan berapa persen matriks yang benar-benar
dari OSRM vs fallback.

In [13]:
EARTH_RADIUS_M = 6_371_000.0


def manhattan_matrix(lats, lons):
    lat_r = np.deg2rad(lats.astype(float))
    lon_r = np.deg2rad(lons.astype(float))
    avg_lat = np.mean(lat_r)
    cos_avg = np.cos(avg_lat)
    dlat = np.abs(lat_r[:, None] - lat_r[None, :])
    dlon = np.abs(lon_r[:, None] - lon_r[None, :]) * cos_avg
    return EARTH_RADIUS_M * (dlat + dlon)


lats_arr = np.array([c[0] for c in coords])
lons_arr = np.array([c[1] for c in coords])
fallback_dist = manhattan_matrix(lats_arr, lons_arr)
fallback_time = fallback_dist / (30_000 / 3600)  # asumsi 30 km/jam, sama seperti geo.py

nan_mask = np.isnan(dist_matrix)
n_fallback_cells = int(nan_mask.sum())
n_total_cells = n_total * n_total

dist_matrix[nan_mask] = fallback_dist[nan_mask]
time_matrix[np.isnan(time_matrix)] = fallback_time[np.isnan(time_matrix)]

np.fill_diagonal(dist_matrix, 0.0)
np.fill_diagonal(time_matrix, 0.0)

pct_fallback = 100 * n_fallback_cells / n_total_cells
print(f"Sel terisi via OSRM : {n_total_cells - n_fallback_cells} ({100 - pct_fallback:.2f}%)")
print(f"Sel fallback Manhattan: {n_fallback_cells} ({pct_fallback:.2f}%)")

Sel terisi via OSRM : 34596 (100.00%)
Sel fallback Manhattan: 0 (0.00%)


## 5. Simpan Output

Simpan ke path yang sama persis dengan yang dicari `backend/app/main.py`
(`DATA_DIR / "distance_matrix.npy"` dan `DATA_DIR / "time_matrix.npy"`).

In [14]:
np.save(OUTPUT_DIR / "distance_matrix.npy", dist_matrix)
np.save(OUTPUT_DIR / "time_matrix.npy", time_matrix)

# Laporan transparansi untuk Bab 5 (persentase sel dari OSRM vs fallback)
report = pd.DataFrame({
    "metric": [
        "n_total_node", "n_total_cells", "n_osrm_cells", "n_fallback_cells", "pct_fallback"
    ],
    "value": [
        n_total, n_total_cells, n_total_cells - n_fallback_cells, n_fallback_cells, round(pct_fallback, 3)
    ],
})
report.to_csv(OUTPUT_DIR / "osrm_matrix_report.csv", index=False)

print(f"Tersimpan: {OUTPUT_DIR}/distance_matrix.npy  (shape {dist_matrix.shape})")
print(f"Tersimpan: {OUTPUT_DIR}/time_matrix.npy  (shape {time_matrix.shape})")
print(f"Tersimpan: {OUTPUT_DIR}/osrm_matrix_report.csv")
print()
print("Langkah berikutnya: copy kedua file .npy ini ke backend/app/data/")
print("lalu restart backend. main.py akan otomatis memuatnya (lihat lifespan()).")

Tersimpan: output_osrm/distance_matrix.npy  (shape (186, 186))
Tersimpan: output_osrm/time_matrix.npy  (shape (186, 186))
Tersimpan: output_osrm/osrm_matrix_report.csv

Langkah berikutnya: copy kedua file .npy ini ke backend/app/data/
lalu restart backend. main.py akan otomatis memuatnya (lihat lifespan()).


## Catatan Penting

- Notebook ini dijalankan **sekali** per pembaruan dataset (bukan tiap request runtime).
  Setelah `distance_matrix.npy` dan `time_matrix.npy` ada di `backend/app/data/`,
  backend Vercel tidak lagi memanggil OSRM sama sekali saat runtime.
- Kalau dataset genangan/depo/IF berubah (titik baru ditambah), notebook ini
  perlu dijalankan ulang dan file `.npy` di-replace, karena urutan dan ukuran
  matriks bergantung pada `n_total` saat ini.
- Kalau server demo publik OSRM sering gagal (rate limit terlalu ketat untuk
  jumlah blok kamu), turunkan `BLOCK_SIZE` atau naikkan `REQUEST_DELAY_S`,
  atau pertimbangkan self-host OSRM lokal untuk menjalankan notebook ini saja
  (tidak perlu self-host permanen, cukup saat notebook ini dijalankan).
- Persentase fallback di `osrm_matrix_report.csv` sebaiknya dilaporkan di Bab 5
  sebagai bagian dari transparansi implementasi (berapa persen matriks yang
  benar-benar jaringan jalan aktual vs pendekatan geometris).